# SIIM-ISIC single-model submission

This notebook creates a Kaggle-ready `submission.csv` from one selected cached model prediction file.

Use the switch in the config cell:

- `SELECTED_MODEL = "b7_joint_meta"`
- `SELECTED_MODEL = "effb0_hybrid14_supcon"`

The notebook does **not** load image checkpoints or re-run CNN inference. It uses the pre-extracted probability caches from the private Kaggle dataset `siim-isic-mean-blend-assets`.


In [ ]:
# Imports
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedShuffleSplit


In [ ]:
# Kaggle paths and run switches
SEED = 42
BLEND_VALID_FRACTION = 0.10
RUN_VALIDATION = True       # Automatically skipped if train cache is absent.

# Change this only.
SELECTED_MODEL = "effb0_hybrid14_supcon"   # options: "b7_joint_meta", "effb0_hybrid14_supcon"

COMP_DIR = Path("/kaggle/input/competitions/siim-isic-melanoma-classification")
ASSET_DIR = Path("/kaggle/input/datasets/omargrist/siim-isic-mean-blend-assets")
FEATURE_DIR = ASSET_DIR / "feature_cache"
WORK_DIR = Path("/kaggle/working")

SAMPLE_SUBMISSION = COMP_DIR / "sample_submission.csv"
SUBMISSION_PATH = WORK_DIR / "submission.csv"

MODEL_CACHES = {
    "b7_joint_meta": {
        "name": "b7_joint_meta",
        "train": FEATURE_DIR / "b7_joint_meta_train_features.npz",
        "test": FEATURE_DIR / "b7_joint_meta_test_features.npz",
    },
    "effb0_hybrid14_supcon": {
        "name": "effb0_hybrid14_supcon",
        "train": FEATURE_DIR / "effb0_hybrid14_supcon_train_features.npz",
        "test": FEATURE_DIR / "effb0_hybrid14_supcon_test_features.npz",
    },
}

print("Selected model:", SELECTED_MODEL)
print("Competition dir:", COMP_DIR)
print("Asset dir:", ASSET_DIR)
print("Feature dir:", FEATURE_DIR)
print("Working dir:", WORK_DIR)


In [ ]:
# Input checks
if SELECTED_MODEL not in MODEL_CACHES:
    raise ValueError(f"Unknown SELECTED_MODEL={SELECTED_MODEL!r}. Choose one of: {list(MODEL_CACHES)}")

spec = MODEL_CACHES[SELECTED_MODEL]
required = [SAMPLE_SUBMISSION, spec["test"]]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required Kaggle input files:{' + '.join(missing)}")

print("Required submission inputs found.")
print(f"Train cache exists: {spec['train'].exists()}")
print(f"Test cache exists:  {spec['test'].exists()}")


In [ ]:
# Cache loading
REQUIRED_CACHE_KEYS = {"image_name", "patient_id", "probs"}


def load_cache(path, split):
    data = np.load(path, allow_pickle=True)
    keys = set(data.files)
    missing = REQUIRED_CACHE_KEYS - keys
    if missing:
        raise KeyError(f"{path.name} missing required keys: {sorted(missing)}. Found: {sorted(keys)}")

    image_name = data["image_name"].astype(str)
    patient_id = data["patient_id"].astype(str)
    probs = data["probs"].astype(np.float32).reshape(-1)

    if len(image_name) != len(probs):
        raise ValueError(f"{path.name}: image_name/probs length mismatch: {len(image_name)} vs {len(probs)}")
    if len(set(image_name)) != len(image_name):
        raise ValueError(f"{path.name}: duplicate image_name values detected")
    if not np.isfinite(probs).all():
        raise ValueError(f"{path.name}: probabilities contain NaN/Inf")
    if probs.min() < -1e-6 or probs.max() > 1 + 1e-6:
        raise ValueError(f"{path.name}: probabilities outside [0, 1]: min={probs.min()}, max={probs.max()}")

    out = pd.DataFrame({
        "image_name": image_name,
        "patient_id": patient_id,
        "target": np.clip(probs, 0.0, 1.0),
    })

    if split == "train":
        if "target" not in keys:
            raise KeyError(f"{path.name}: train cache is missing target")
        y = data["target"].astype(int).reshape(-1)
        if len(y) != len(out):
            raise ValueError(f"{path.name}: target/probs length mismatch")
        out["label"] = y

    return out


In [ ]:
# Build and save the selected-model submission
sample_submission = pd.read_csv(SAMPLE_SUBMISSION)
print("Sample submission:", sample_submission.shape, list(sample_submission.columns))

if list(sample_submission.columns) != ["image_name", "target"]:
    raise ValueError(f"Unexpected sample_submission columns: {list(sample_submission.columns)}")

test_pred = load_cache(spec["test"], split="test")
submission = sample_submission[["image_name"]].merge(
    test_pred[["image_name", "target"]],
    on="image_name",
    how="left",
)

if submission["target"].isna().any():
    missing = submission.loc[submission["target"].isna(), "image_name"].head().tolist()
    raise ValueError(f"Missing predictions for sample_submission rows, examples={missing}")
if len(submission) != len(sample_submission):
    raise ValueError("Submission row count changed during merge")
if list(submission.columns) != ["image_name", "target"]:
    raise ValueError(f"Submission columns are wrong: {list(submission.columns)}")
if not submission["target"].between(0, 1).all():
    raise ValueError("Submission target values must be in [0, 1]")

# Kaggle expects this exact output name in /kaggle/working.
submission.to_csv(SUBMISSION_PATH, index=False)
submission.to_csv(WORK_DIR / f"submission_{SELECTED_MODEL}.csv", index=False)

print(f"Saved Kaggle submission: {SUBMISSION_PATH}")
print(f"Saved copy: {WORK_DIR / f'submission_{SELECTED_MODEL}.csv'}")
print(submission.head())
print(submission["target"].describe())


In [ ]:
# Validation helpers

def split_validation(meta):
    patients = meta.groupby("patient_id")["label"].max().reset_index()
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=BLEND_VALID_FRACTION,
        random_state=SEED,
    )
    _, val_idx = next(splitter.split(patients["patient_id"], patients["label"]))
    val_patients = set(patients.iloc[val_idx]["patient_id"])
    return meta["patient_id"].isin(val_patients).values


def evaluate_predictions(y_true, y_prob, name, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "model": name,
        "threshold": float(threshold),
        "auc": roc_auc_score(y_true, y_prob),
        "auprc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }

    print(f"\n{name} @ threshold={threshold:.4f}")
    for key in ["auc", "auprc", "accuracy", "f1"]:
        print(f"{key}: {metrics[key]:.5f}")
    print(classification_report(y_true, y_pred, target_names=["Benign", "Melanoma"]))

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm)
    ax.set_xticks([0, 1], labels=["Benign", "Melanoma"])
    ax.set_yticks([0, 1], labels=["Benign", "Melanoma"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix - {name}")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(WORK_DIR / f"confusion_matrix_{name}.png", dpi=150)
    plt.show()

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f"AUC = {metrics['auc']:.4f}")
    ax.plot([0, 1], [0, 1], "--", linewidth=1)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC curve - {name}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(WORK_DIR / f"roc_curve_{name}.png", dpi=150)
    plt.show()

    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(recall, precision, label=f"AUPRC = {metrics['auprc']:.4f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"Precision-recall curve - {name}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(WORK_DIR / f"pr_curve_{name}.png", dpi=150)
    plt.show()

    return metrics


def find_best_threshold(y_true, y_prob):
    rows = []
    for th in np.linspace(0.01, 0.99, 999):
        pred = (y_prob >= th).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        precision = tp / max(1, tp + fp)
        recall = tp / max(1, tp + fn)
        rows.append({
            "threshold": th,
            "precision": precision,
            "recall": recall,
            "f1": f1_score(y_true, pred),
            "tp": tp,
            "fp": fp,
            "tn": tn,
            "fn": fn,
        })

    df = pd.DataFrame(rows)
    return df.sort_values("f1", ascending=False).iloc[0], df


In [ ]:
# Optional validation run for the selected model
if RUN_VALIDATION and spec["train"].exists():
    train_pred = load_cache(spec["train"], split="train")
    val_mask = split_validation(train_pred)

    val_df = train_pred[val_mask].reset_index(drop=True)
    y_val = val_df["label"].values.astype(int)
    val_probs = val_df["target"].values.astype(np.float32)

    metrics_05 = evaluate_predictions(
        y_val,
        val_probs,
        name=f"{SELECTED_MODEL}_threshold_0_5",
        threshold=0.5,
    )

    valid_pred = val_df[["image_name", "patient_id", "label", "target"]].rename(
        columns={"label": "true_label", "target": "pred"}
    )
    valid_pred.to_csv(WORK_DIR / f"valid_predictions_{SELECTED_MODEL}.csv", index=False)

    best_row, threshold_df = find_best_threshold(y_val, val_probs)
    print("\nBest threshold row:")
    print(best_row)
    threshold_df.to_csv(WORK_DIR / f"threshold_sweep_{SELECTED_MODEL}.csv", index=False)

    metrics_best = evaluate_predictions(
        y_val,
        val_probs,
        name=f"{SELECTED_MODEL}_best_threshold",
        threshold=float(best_row["threshold"]),
    )

    pd.DataFrame([metrics_05, metrics_best]).to_csv(
        WORK_DIR / f"validation_metrics_{SELECTED_MODEL}.csv",
        index=False,
    )
else:
    print("Validation skipped. Set RUN_VALIDATION=True and upload the selected train feature cache to enable it.")


## Submission notes

- Submit `/kaggle/working/submission.csv`.
- The notebook also writes `submission_<selected_model>.csv` for clarity.
- Do not threshold the test predictions: SIIM-ISIC submission uses probabilities in the `target` column.
